# 🛣 Road Damage Detection — YOLOv8 Fine-tuning
### Free GPU training on Google Colab
1. Click **Runtime → Run All**
2. After training (~15 min), download the ONNX model
3. Place it in `models/YOLOv8_RDD_Trained.onnx`

In [ ]:
# 1. Install dependencies
!pip install ultralytics roboflow onnx onnxruntime -q

In [ ]:
# 2. Download dataset from Roboflow
from roboflow import Roboflow

API_KEY = "DxjqJKaL6LQgshUa7gi7"  # Your Roboflow API key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("rdd").project("objdetect10001-with-manholes-pcjte")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# 3. Train YOLOv8 Nano (auto GPU/CPU)
from ultralytics import YOLO
import torch

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Find data.yaml
import os
yaml_path = None
for root, dirs, files in os.walk(dataset.location):
    if "data.yaml" in files:
        yaml_path = os.path.join(root, "data.yaml")
        break

print(f"Using data.yaml: {yaml_path}")

# Auto-detect GPU/CPU
device = 0 if torch.cuda.is_available() else "cpu"
batch = 16 if device == 0 else 8
print(f"Device: {device} | Batch: {batch}")

if device == "cpu":
    print(" No GPU! Go to Runtime > Change runtime type > T4 GPU for 10x faster training!")

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=batch,
    name="YOLOv8_RDD",
    exist_ok=True,
    patience=8,
    device=device,
)

In [ ]:
# 2.5 Fix data.yaml paths (Roboflow bug workaround)
import yaml

# Find and fix the data.yaml
yaml_path = None
dataset_root = None
for root, dirs, files in os.walk(dataset.location):
    if "data.yaml" in files:
        yaml_path = os.path.join(root, "data.yaml")
        dataset_root = root
        break

print(f"data.yaml found at: {yaml_path}")

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print(f"Before: train={cfg.get('train')}, val={cfg.get('val')}, test={cfg.get('test')}")

# Fix paths to be absolute
cfg['path'] = dataset_root
cfg['train'] = 'train/images'
cfg['val'] = 'train/images'   # Use train for validation too
cfg['test'] = 'train/images'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print(f"After: path={cfg['path']}, train={cfg['train']}")

# Quick verify
import glob
imgs = glob.glob(f"{dataset_root}/train/images/*.jpg") + glob.glob(f"{dataset_root}/train/images/*.png")
print(f"Images found: {len(imgs)}")

In [ ]:
# 4. Export to ONNX
import shutil
from pathlib import Path

best_pt = Path(results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best_pt))

# Export ONNX
model.export(format="onnx", opset=12, simplify=True, imgsz=640, dynamic=False)

# Show output shape
import onnxruntime as ort
onnx_path = best_pt.with_suffix(".onnx")
session = ort.InferenceSession(str(onnx_path))
out_shape = session.get_outputs()[0].shape
n_classes = out_shape[1] - 4
print(f"\nModel ready: {onnx_path}")
print(f"Size: {onnx_path.stat().st_size / 1e6:.1f} MB")
print(f"Output: {out_shape}  Classes: {n_classes}")
print("\nDownload the ONNX file from the Files sidebar on the left!")